# Code volume: qarp vs five stacks

Three quantum-chemistry algorithms — VQE, SS-VQE, ADAPT-VQE — written once per stack.
The stacks split into two groups that answer different questions: **primitive** stacks
(qulacs, cirq, qiskit_raw) ship no algorithm layer, so the user writes the plumbing;
**frameworks** (pennylane, qiskit-nature) ship the algorithm itself.

Every implementation must return the same number, checked against an independent oracle,
before any lines are counted.  Protocol, fairness rules and discussion are in `README.md`.

Deps: `pip install -e ".[notebooks,integrations,bench]"` plus
`pyscf openfermionpyscf qiskit-nature qiskit-algorithms`.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import HTML, Markdown, display

from run import ALGORITHMS, IMPL, STACKS, check, loc_rows, run_all

PRIMITIVE = [s for s, (_, table) in STACKS.items() if table == "primitive"]
FRAMEWORK = [s for s, (_, table) in STACKS.items() if table == "framework"]


def side_by_side(algorithm, stacks):
    """qarp against each named stack, source next to source."""
    cell = "<td style='vertical-align:top'><b>{}</b><pre style='font-size:85%'>{}</pre></td>"
    columns = [("qarp", IMPL / "qarp" / f"{algorithm}.py")]
    columns += [(s, IMPL / s / f"{algorithm}.py") for s in stacks]
    html = "".join(cell.format(n, p.read_text()) for n, p in columns if p.exists())
    display(HTML(f"<table><tr>{html}</tr></table>"))

## The code

Each script is complete and runnable as-is.

### VQE — H4 chain, STO-3G, UCCSD, 26 parameters

In [ ]:
side_by_side("vqe", PRIMITIVE)

In [ ]:
side_by_side("vqe", FRAMEWORK)

### SS-VQE — H2, STO-3G, hardware-efficient ansatz ×6, three states

In [ ]:
side_by_side("ssvqe", PRIMITIVE)

In [ ]:
side_by_side("ssvqe", FRAMEWORK)

### ADAPT-VQE — LiH, STO-3G, (2e, 3o) active space

In [ ]:
side_by_side("adapt_vqe", PRIMITIVE)

In [ ]:
side_by_side("adapt_vqe", FRAMEWORK)

### Shot-based VQE — H2, STO-3G, UCCSD, qubit-wise-commuting grouping

The row the amortised figure was always understating: one argument on the qarp side,
a grouping partition + basis rotation + sampler + estimator on every primitive stack.

In [ ]:
side_by_side("vqe_shots", PRIMITIVE)

In [ ]:
side_by_side("vqe_shots", FRAMEWORK)

### QPE — two-qubit commuting Hamiltonian, 4 ancillas

The terms commute, so the Trotter step is exact and the phase 11/16 is exactly
representable: every stack returns it with probability 1, and the oracle is analytic.

In [ ]:
side_by_side("qpe", PRIMITIVE)

In [ ]:
side_by_side("qpe", FRAMEWORK)

### QAOA — MaxCut on a 6-vertex ring, p = 1

Oracle is the closed form: <C> = n/2 + (n/4) sin(4b) sin(2g), maximum 3n/4.

In [ ]:
side_by_side("qaoa", PRIMITIVE)

In [ ]:
side_by_side("qaoa", FRAMEWORK)

### What each primitive stack has to write first

`common.py` is what a user of that stack writes before any variational algorithm runs:
the molecular Hamiltonian, the UCC excitation pool, the parametric ansatz with its
parameter-sharing layout, the reference state.  It is written once and counted once in
the amortised row.  The frameworks have no `common.py` — that is the point of them.

In [ ]:
for stack in PRIMITIVE:
    display(Markdown(f"#### {stack}"))
    display(Markdown("```python\n" + (IMPL / stack / "common.py").read_text() + "\n```"))

## Lines of code

Physical lines, excluding blanks, comments and docstrings; imports count everywhere.
Per primitive stack: the script, the script plus all of `common.py`, and the script plus
only the `common.py` functions it actually reaches — the number a sceptical reader computes.

In [ ]:
def loc_frame(table):
    rows, competitors = loc_rows(table)
    out = []
    for row in rows:
        record = {"algorithm": row["algorithm"], "qarp": row["qarp"]}
        for stack in competitors:
            cell = row[stack]
            if cell is None:
                record[stack] = None
                continue
            script, full, reached = cell
            record[stack] = script
            if full:
                record[f"{stack} +common"] = script + full
                record[f"{stack} +reached"] = script + reached
        out.append(record)
    return pd.DataFrame(out).set_index("algorithm")


display(Markdown("### vs primitive stacks"))
display(loc_frame("primitive"))
display(Markdown("### vs frameworks"))
display(loc_frame("framework"))

## Run everything

Each script runs as a subprocess in its own directory.  **This is slow**: cirq's simulator
and PennyLane's `default.qubit` need hundreds of ms per energy evaluation against qulacs'
few ms, so the H4 VQE alone is tens of minutes on those stacks.  Pass `stacks=` or
`algorithms=` to `run_all` for a subset.

In [ ]:
results = run_all(stacks=["qarp", "qulacs"])

rows = []
for algorithm, cells in results.items():
    key = ALGORITHMS[algorithm]["key"]
    for stack, values in cells.items():
        rows.append({"algorithm": key, "stack": stack, "result": values[key],
                     "oracle": values.get("exact", values.get("CCSD"))})
display(pd.DataFrame(rows).set_index(["algorithm", "stack"]))

failures = check(results)
assert not failures, failures
print("every stack agrees within its table's tolerance")

## Capability coverage

What a line count cannot say.  Every cell is resolved from the live API by importing a
candidate symbol; the candidates live in `capabilities.py`, so a wrong one is a visible
bug rather than an invisible claim.  A `--` means *not found at any probed path*.

In [ ]:
from capabilities import CAPABILITIES, STACKS, found

coverage = pd.DataFrame(
    [["yes" if found(c, s) else "--" for s in STACKS] for c in CAPABILITIES],
    index=list(CAPABILITIES),
    columns=list(STACKS),
)
display(coverage)
print(coverage.eq("yes").sum().to_string())